# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iqralodhi7866/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import pandas as pd
import duckdb
import google.colab.userdata

# 1. Authenticate Hugging Face
hf_token = google.colab.userdata.get('HF_TOKEN')
duckdb.sql("DROP SECRET IF EXISTS __default_huggingface;")
duckdb.sql(f"""
    CREATE SECRET (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")

# 2. Load dataset and create baseline action score queue (fixed column name)
query = """
SELECT
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    CASE
        WHEN gsc_avg_position < 15 AND gsc_impressions > 100 THEN 'REFRESH_STALE_CONTENT'
        ELSE 'CTR_OPTIMIZATION_NEEDED'
    END as reason_code,
    (COALESCE(gsc_impressions, 0) * 0.5 + COALESCE(gsc_clicks, 0) * 2.0) as action_score,
    CASE
        WHEN gsc_avg_position < 15 AND gsc_impressions > 100 THEN 'Refresh Page Content'
        else 'Optimize Title & Meta'
    END as action_label
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_impressions IS NOT NULL AND gsc_impressions IS TRUE
LIMIT 100;
"""

df_queue = duckdb.sql(query).df()

# 3. Ensure output directory exists and write CSV
os.makedirs('work/outputs', exist_ok=True)
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Baseline action score queue generated successfully!")
display(df_queue.head(5))

Baseline action score queue generated successfully!


,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,reason_code,action_score,action_label
0,content_b7e512995f79d5a6,2026-03-01,20,0,3.350000,CTR_OPTIMIZATION_NEEDED,10.0,Optimize Title & Meta
1,content_05597932fe4da067,2026-03-01,1,0,0.000000,CTR_OPTIMIZATION_NEEDED,0.5,Optimize Title & Meta
2,content_7a105f548d9c6916,2026-03-01,125,1,4.928000,REFRESH_STALE_CONTENT,64.5,Refresh Page Content
3,content_905aa32a0230694e,2026-03-01,7,0,4.000000,CTR_OPTIMIZATION_NEEDED,3.5,Optimize Title & Meta
4,content_a3ea9792f793ec72,2026-03-01,11,0,2.272727,CTR_OPTIMIZATION_NEEDED,5.5,Optimize Title & Meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.